# Initializing packages 

In [ ]:
import os
import json
from dotenv import load_dotenv
from typing import Annotated
from langchain_core.messages import HumanMessage, BaseMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict
from langchain_mcp_adapters.client import MultiServerMCPClient

from agentevals.trajectory.match import create_trajectory_match_evaluator

load_dotenv()  # loads BLABLADOR_API_KEY from .env

# DeepEval tracing imports
# from deepeval.tracing import observe, update_current_span, update_current_trace  #for production monitoring
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
#from deepeval.tracing import get_trace_stack
from deepeval import evaluate
from deepeval.dataset import Golden
from deepeval.models import LiteLLMModel

from deepeval.metrics import (
    # RAG / Retrieval metrics
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    ContextualRelevancyMetric,

    # Agent / Tool metrics
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    TaskCompletionMetric,
    step_efficiency,

    # General quality
    HallucinationMetric,
    MisuseMetric,
)
from deepeval.evaluate.configs import AsyncConfig

In [2]:
BLABLADOR_BASE_URL = "https://api.helmholtz-blablador.fz-juelich.de/v1"
api_key=os.getenv("BLABLADOR_API_KEY")

In [3]:
from blablador import Models 

models = Models(api_key).get_model_ids()
models

['alias-apertus',
 '15 - Apertus-8B-Instruct-2509 - A new swiss model from September 2025',
 'alias-eve',
 '20 - EVE-Instruct - Expert Earth Observation and Earth Science (ES) domains',
 '01 - GPT-OSS-120b - an open model released by OpenAI in August 2025',
 'alias-fast',
 '01 - MiniMax-M2.7 - our best model as of April, 2026',
 'alias-huge',
 '02 - Qwen3.5-122B-A10B-FP8, general purpose large model',
 'alias-large',
 'alias-code',
 '09 - Qwen3-Coder-Next-FP8 from Feb 2026',
 'alias-embeddings',
 'gpt-3.5-turbo',
 'text-davinci-003',
 'text-embedding-ada-002',
 'alias-qwen3-8b-embeddings',
 '08 - Qwen3.6-35B-A3B-FP8 - Multimodal model from Apr 2026',
 'alias-qwen36-35b',
 '10 - Muse Glimmer 30b - the newest META model as of August 11, 2026',
 '10 - Qwen3.5-397B-A17B',
 '90 - MiniMax-M3-AWQ-INT4 (strube1-booster)',
 'DeepSeek-V4-Flash-0731',
 'Kimi-K3-1M',
 'alias-deepseek-v4-flash-0731',
 'alias-kimi-k3-1m',
 'alias-minimax-m3-awq-int4',
 'alias-muse',
 'alias-qwen-huge',
 'faster-whis

# Agent with Tools via MCP server Initilization

In [25]:
#React-agent
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    
with open("mcp_config.json") as f:
    config = json.load(f)

    client = MultiServerMCPClient(config)
    tools = await client.get_tools()
    print(f"Loaded {len(tools)} tools: {[t.name for t in tools]}")

    llm = ChatOpenAI(
        model="01 - MiniMax-M2.7 - our best model as of April, 2026",
        base_url=BLABLADOR_BASE_URL,
        api_key=api_key,
    )
    llm_with_tools = llm.bind_tools(tools)
    print("Model ready.")

    async def agent_node(state: AgentState) -> AgentState:
        response = await llm_with_tools.ainvoke(state["messages"])  # must await
        return {"messages": [response]}

    def build_graph():
        graph = StateGraph(AgentState)
        graph.add_node("agent", agent_node)
        graph.add_node("tools", ToolNode(tools))
        graph.add_edge(START, "agent")
        graph.add_conditional_edges("agent", tools_condition)
        graph.add_edge("tools", "agent")
        return graph.compile()

    app = build_graph()
    print("Graph compiled successfully.")


Loaded 2 tools: ['search_UFZ_guidelines', 'search_funding_guidelines']
Model ready.
Graph compiled successfully.


# Agent execution function that extracts actual output, retrieval context, and tools called for evaluation


In [26]:

async def run_agent(user_input: str) -> dict:
    result = await app.ainvoke({"messages": [HumanMessage(content=user_input)]})
    messages = result["messages"]
    
    # 1. actual_output — last AIMessage that has content (not a tool call)
    actual_output = next(
        msg.content
        for msg in reversed(messages)
        if isinstance(msg, AIMessage) and msg.content
    )

    # 2. retrieval_context — what your tools returned
    retrieval_context = [
        str(msg.content) for msg in messages
        if isinstance(msg, ToolMessage)
    ]

    # 3. tools_called — pair each AIMessage tool call with its ToolMessage output
    tools_called = []
    tool_outputs = [msg for msg in messages if isinstance(msg, ToolMessage)]
    tool_idx = 0

    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                output = tool_outputs[tool_idx].content if tool_idx < len(tool_outputs) else ""
                tools_called.append(ToolCall(
                    name=tc["name"],
                    input_parameters=tc.get("args") or {},   # <-- renamed + defensive fallback
                    output=str(output),
                ))
                tool_idx += 1

    return {
        "actual_output": actual_output,
        "retrieval_context": retrieval_context,
        "tools_called": tools_called,
        "messages": messages,
    }

In [ ]:
queries = [
    "What is (123 * 456) + 789?",
    "What are the UFZ guidelines for long-term archiving?",
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {await run_agent(q)}")   # must await async run()
    print()

# Judge for evaluation metrics
#### MoE variants (same architecture):
#### Qwen/Qwen3.5-35B-A3B — 35B total, 3B active
#### Qwen/Qwen3.5-122B-A10B — 122B / 10B active
#### Qwen/Qwen3.5-397B-A17B — 397B / 17B active

In [27]:

judge_small = LiteLLMModel(
    model= "openai/08 - Qwen3.6-35B-A3B-FP8 - Multimodal model from Apr 2026",          # prefix with "openai/" for OpenAI-compat APIs
    base_url=BLABLADOR_BASE_URL,
    api_key=os.getenv("BLABLADOR_API_KEY")
)

In [30]:
judge = LiteLLMModel(
    model= "openai/08 - Qwen3.6-35B-A3B-FP8 - Multimodal model from Apr 2026",          # prefix with "openai/" for OpenAI-compat APIs
    base_url=BLABLADOR_BASE_URL,
    api_key=os.getenv("BLABLADOR_API_KEY")
)

In [29]:
judge_large = LiteLLMModel(
    model= "openai/10 - Qwen3.5-397B-A17B",          # prefix with "openai/" for OpenAI-compat APIs
    base_url=BLABLADOR_BASE_URL,
    api_key=os.getenv("BLABLADOR_API_KEY")
)

# Metrics configrations
#### A metric is only successful if the evaluation score is equal to or greater than threshold, which is defaulted to 0.5 for all metrics.

In [ ]:
tool_correctness = ToolCorrectnessMetric( threshold=0.5, include_reason=True,)
argument_correctness = ArgumentCorrectnessMetric(threshold=0.5, model=judge, include_reason=True,)
task_completion = TaskCompletionMetric(threshold=0.5, model=judge, include_reason=True)
faithfulness    = FaithfulnessMetric(threshold=0.5, model=judge, include_reason=True)
answer_rel      = AnswerRelevancyMetric(threshold=0.5, model=judge, include_reason=True)
hallucination   = HallucinationMetric(threshold=0.5, model=judge, include_reason=True)
contextual_recall = ContextualRecallMetric(threshold=0.5, model=judge, include_reason=True)
contextual_precision = ContextualPrecisionMetric(threshold=0.5, model=judge, include_reason=True)
contextual_relevancy = ContextualRelevancyMetric(threshold=0.5, model=judge, include_reason=True)
misuse = MisuseMetric(threshold=0.5, model=judge,domain="research data management", include_reason=True)

In [ ]:
# for test_result in results.test_results:
#     for metric_data in test_result.metrics_data:
#         print(metric_data.name, metric_data.score, metric_data.reason)

In [ ]:
# when change something in test_cases.py, reload it to reflect changes in the notebook
import importlib, middle_test_cases
importlib.reload(middle_test_cases)

In [13]:
from middle_test_cases import TEST_CASES, build_test_case

In [14]:
def build_query(tc: dict) -> str:
    if "{source_extract}" in tc["input_template"]:
        return tc["input_template"].format(source_extract=tc.get("source_extract", ""))
    return tc["input_template"]

test_cases = []
i = 0
for i, tc in enumerate(TEST_CASES):
    print(i + 1)
    query = build_query(tc)
    result = await run_agent(query)
    if result["actual_output"].startswith("[NO FINAL RESPONSE"):
        print(f"⚠ {tc['id']}: agent produced no final AIMessage content")
    test_case = build_test_case(tc, result)
    test_cases.append(test_case)

1
2
3
4
5
6
7
8
9
10
11
12


### Save built test cases into json

In [ ]:
def toolcall_to_dict(tc: ToolCall) -> dict:
    return {
        "name": tc.name,
        "input_parameters": tc.input_parameters or {},
        "output": tc.output,
    }

def testcase_to_dict(tc: LLMTestCase, tc_id: str = None, tc_type: str = None) -> dict:
    return {
        "id": tc_id,
        "type": tc_type,
        "input": tc.input,
        "actual_output": tc.actual_output,
        "expected_output": tc.expected_output,
        "context": tc.context,
        "retrieval_context": tc.retrieval_context,
        "tools_called": [toolcall_to_dict(t) for t in (tc.tools_called or [])],
        "expected_tools": [toolcall_to_dict(t) for t in (tc.expected_tools or [])],
    }

with open("test_cases_built2.json", "w", encoding="utf-8") as f:
    json.dump(
        [testcase_to_dict(tc, tc_id=t["id"], tc_type=t["type"])
         for tc, t in zip(test_cases, TEST_CASES)],
        f, indent=2, ensure_ascii=False
    )

print(f"Saved {len(test_cases)} built test cases to test_cases_built2.json")

Saved 12 built test cases to test_cases_built2.json


In [20]:
results = evaluate(
    test_cases=test_cases,
    metrics=[
        tool_correctness,
        argument_correctness,
        task_completion,
        faithfulness,
        answer_rel,
        hallucination,
        contextual_recall,
        contextual_precision,
        contextual_relevancy,
        misuse
    ],
    async_config=AsyncConfig(run_async=False),
)

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Argument Correctness Metric! (using openai/alias-fast (('alias-fast', 'openai',
None, None)), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Task Completion Metric! (using openai/alias-fast (('alias-fast', 'openai', 
None, None)), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/alias-fast (('alias-fast', 'openai', None, 
None)), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/alias-fast (('alias-fast', 'openai', 
None, None)), strict=False, async_mode=False)...

...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai/alias-fast (('alias-fast', 'openai',
None, None)), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using openai/alias-fast (('alias-fast', 'openai',
None, None)), strict=False, async_mode=False)...

✨ You're running DeepEval's latest Misuse Metric! (using openai/alias-fast (('alias-fast', 'openai', None, None)),
strict=False, async_mode=False)...

Output()

TimeoutError: call timed out after 88.5s (per attempt). Increase DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE (None disables) or reduce work per attempt.

In [17]:
def serialize_result(result, tc_id):
    entry = {"id": tc_id, "metrics": []}
    for test_result in result.test_results:
        for metric_data in test_result.metrics_data:
            entry["metrics"].append({
                "name": metric_data.name,
                "score": metric_data.score,
                "success": metric_data.success,
                "reason": metric_data.reason,
            })
    return entry

In [ ]:
from time import sleep

all_results = []
tc_witherror = []

def evaluate_test_cases(tc_id, tc):
    print(f"Running {tc_id}...")
    try:

        result = evaluate(
            test_cases=[tc],
            metrics=[
                faithfulness, answer_rel, hallucination,
                tool_correctness, argument_correctness, task_completion,
                contextual_recall, contextual_precision, contextual_relevancy,
                misuse,
            ],
            async_config=AsyncConfig(run_async=False),
        )

        serialized = serialize_result(result, tc_id)
        all_results.append(serialized)


    except Exception as e:
        print(f"  ✗ {tc_id} failed with error: {e}")
        tc_witherror.append(tc_id)
        sleep(65)  # optional: wait a bit before continuing to avoid rapid-fire errors
        return

    print(f"  ✓ {tc_id} done — {len(all_results)}/{len(loaded_cases)} saved so far")



In [ ]:
all_results = []

try:
    for tc_id, tc in loaded_cases:
        evaluate_test_cases(tc_id, tc)
        
    if tc_witherror:
        print(f"\nThe following test cases failed with errors: {tc_witherror}")
        for tc_id, tc in loaded_cases:
            if tc_id in tc_witherror:
                evaluate_test_cases(tc_id, tc)
                print(f"  - {tc_id}: {tc}")
                tc_witherror.remove(tc_id)  # remove from the list after re-evaluation

except Exception as e:
    print(f"An unexpected error occurred: {e}")

finally:
    # save after every case — so a crash on case 8 doesn't lose 1-7
    filename = "all_results.json"
    counter = 1
    while True:
        try:
            filename = f"all_results_{counter}.json"
            with open(filename, "x", encoding="utf-8") as f:
                json.dump(all_results, f, indent=4, default=str)
            print(f"  ✓ {tc_id} results saved to {filename}")
            break
        except FileExistsError:
            counter += 1


print("\nAll done. Final file: all_results.json")

An unexpected error occurred: too many values to unpack (expected 2)


NameError: name 'tc_id' is not defined

In [ ]:
def dict_to_toolcall(d: dict) -> ToolCall:
    return ToolCall(
        name=d["name"],
        input_parameters=d.get("input_parameters") or {},
        output=d.get("output"),
    )

def dict_to_testcase(d: dict) -> LLMTestCase:
    return LLMTestCase(
        input=d["input"],
        actual_output=d["actual_output"],
        expected_output=d.get("expected_output"),
        context=d.get("context"),
        retrieval_context=d.get("retrieval_context"),
        tools_called=[dict_to_toolcall(t) for t in d.get("tools_called", [])],
        expected_tools=[dict_to_toolcall(t) for t in d.get("expected_tools", [])],
    )

with open("test_cases_built.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# keep id + LLMTestCase paired together
loaded_cases = [(d["id"], dict_to_testcase(d)) for d in raw]

print(f"Loaded {len(loaded_cases)} test cases")